In [2]:
import numpy as np 
import pandas as pd 

In [3]:
df = pd.read_csv('traffic_logs.csv')

In [4]:
#checking for missing values in the dataset and where they are located

In [5]:
df.isna().sum()

Timestamp          0
Vehicle_Plate      0
Recorded_Speed    15
Base_Toll          0
Vehicle_Type       0
dtype: int64

In [6]:
l=df[df['Recorded_Speed'].isna()].index

In [7]:
df2=df.loc[l]


In [8]:
#isolating the rows with missing values in the 'Recorded_Speed' column and storing them in a new dataframe df2 for further analysis.

In [9]:
df2

,Timestamp,Vehicle_Plate,Recorded_Speed,Base_Toll,Vehicle_Type
31,2026-06-01 07:45:00,IL-7873,NaN,15.0,SUV
286,2026-06-03 23:30:00,IL-5175,NaN,10.0,Sedan
294,2026-06-04 01:30:00,FL-8575,NaN,12.0,Truck
345,2026-06-04 14:15:00,CA-6115,NaN,12.0,EV
448,2026-06-05 16:00:00,CA-2248,NaN,15.0,Sedan
539,2026-06-06 14:45:00,NY-8970,NaN,10.0,Sedan
551,2026-06-06 17:45:00,NY-2409,NaN,12.0,SUV
677,2026-06-08 01:15:00,IL-2290,NaN,10.0,SUV
691,2026-06-08 04:45:00,NY-7183,NaN,10.0,Truck
699,2026-06-08 06:45:00,IL-1556,NaN,12.0,Sedan


In [10]:
#Filling missing values in the 'Recorded_Speed' column with the median speed of each vehicle type

In [11]:
s=df.groupby('Vehicle_Type')['Recorded_Speed'].median()

In [12]:
i=s.index

In [13]:
for i in df.iloc[l]['Vehicle_Type']:
    if df['Recorded_Speed'].isna().any():
         df['Recorded_Speed'].fillna(s[i], inplace=True)


C:\Users\Mahrab Khan\AppData\Local\Temp\ipykernel_5960\1226846561.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Recorded_Speed'].fillna(s[i], inplace=True)


In [14]:
df.isna().sum()

Timestamp          0
Vehicle_Plate      0
Recorded_Speed    15
Base_Toll          0
Vehicle_Type       0
dtype: int64

In [15]:
df['Is_Anomaly'] = np.where((df['Recorded_Speed']>110) & (df['Recorded_Speed']<5), True, False)

In [16]:
#To add the 'Final_toll' column where we make the nessesary changes in th bass toll according to the rush hour

In [17]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [22]:
df['Final_toll'] = np.nan
for idx in df.index:
    h=df['Timestamp'].dt.hour[idx]
    if ((h >= 6) & (h <= 9)) | ((h >= 16) & (h <= 19)):
        df.at[idx, 'Final_toll'] = df.at[idx, 'Base_Toll'] * 1.5
    else:
        df.at[idx, 'Final_toll'] = df.at[idx, 'Base_Toll']

In [28]:
df['Is_Volition'] = np.where(df['Recorded_Speed']>80, True, False)

In [32]:
state=[]
a=''
for i in df.index:
    vp=df.iloc[i]['Vehicle_Plate']
    for j in vp:
        if j.isalpha():
            a+=j
    if a not in state:
        state.append(a)
    a=''
        
print(state)

['FL', 'TX', 'IL', 'NY', 'CA']


In [33]:
df['State']= df['Vehicle_Plate'].str.extract(r'([A-Z]+)')

In [35]:
df.groupby('State')['Final_toll'].sum().sort_values(ascending=False)

State
NY    2958.0
CA    2785.0
IL    2720.5
FL    2672.0
TX    2627.5
Name: Final_toll, dtype: float64